# Loading one constituency

This example loads C01 from the project's synthetic CSV files, validates the source tables, and explores the returned voter, candidate, network, and event data. It prepares data for a simulation but does not simulate voter behavior.

## 1. Find the project files

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
project = cwd if (cwd / 'data').exists() else cwd.parent
module_dir = project / 'constituency loader'
data_dir = project / 'data' / 'synthetic_election_dataset'
sys.path.insert(0, str(module_dir))

print('Project folder:', project)
print('Data folder exists:', data_dir.exists())

Project folder: c:\Users\PUSHKAR\Desktop\election-dynamics-simulator
Data folder exists: True


## 2. Load C01
The loader checks the source tables, selects C01's voters and candidates, keeps only edges internal to C01, and includes events scoped to C01, its region, or the state.

In [2]:
from loader import load_constituency

constituency = load_constituency(data_dir, constituency_id=1)
constituency['diagnostics']

{'constituency_id': 1,
 'region_id': 1,
 'voter_count': 3000,
 'candidate_count': 3,
 'internal_edge_count': 24080,
 'excluded_boundary_edge_count': 393,
 'event_count': 98}

## 3. Inspect the four tables
Each item below is a pandas DataFrame that a simulation can use as input.

In [3]:
voters = constituency['voters']
candidates = constituency['candidates']
edges = constituency['social_network']
events = constituency['events']

print('Voters:', voters.shape)
display(voters.head(3))
print('Candidates:', candidates.shape)
display(candidates)
print('Internal directed edges:', edges.shape)
display(edges.head(3))
print('Relevant events:', events.shape)
display(events[['event_id', 'region_id', 'constituency_id', 'event_day', 'source_type', 'issue']].head(5))

Voters: (3000, 20)


,voter_id,constituency_id,region_id,age_group,gender,education_level,wealth_level,occupation,urban_rural,initial_opinion,event_sensitivity,turnout_propensity,susceptibility,influence_power,confidence_threshold,issue_employment,issue_economy,issue_education,issue_healthcare,issue_agriculture
0,C01_V0001,1,1,25-34,Female,Primary,Low,Homemaker,Urban,-0.7741,0.8520,0.1807,0.3241,0.5093,0.7070,0.6792,0.3476,0.5386,0.5670,0.4226
1,C01_V0002,1,1,45-59,Male,Primary,Middle,Unskilled labourer,Rural,0.5900,0.5034,0.4646,0.6524,0.3129,0.1424,0.6790,0.4237,0.4188,0.6829,0.7025
2,C01_V0003,1,1,45-59,Male,Secondary,Middle,Private employee,Urban,-0.3830,0.5008,0.7371,0.2184,0.2397,0.4956,0.8527,0.8341,0.3470,0.6241,0.6819


Candidates: (3, 13)


,candidate_id,constituency_id,region_id,party_id,party_name,is_incumbent,experience_level,local_popularity,position_employment,position_economy,position_education,position_healthcare,position_agriculture
0,CAND_0001,1,1,P1,Progressive Alliance,False,2,0.7790,-0.9431,-0.1499,-1.0000,-0.4155,-0.0608
1,CAND_0002,1,1,P2,National Unity Party,False,1,0.2420,0.0334,0.0979,-0.2211,0.2022,0.1545
2,CAND_0003,1,1,P3,People's Front,False,1,0.6095,0.2069,0.6307,0.9925,0.5636,0.7224


Internal directed edges: (24080, 7)


,edge_id,constituency_id,region_id,source_voter_id,target_voter_id,edge_weight,relationship_type
0,E0000001,1,1,C01_V0578,C01_V0001,0.5234,Friends
1,E0000002,1,1,C01_V0731,C01_V0001,0.6259,Colleagues
2,E0000003,1,1,C01_V1066,C01_V0001,0.8716,Colleagues


Relevant events: (98, 14)


,event_id,region_id,constituency_id,event_day,source_type,issue
0,EVT_00002,1,1,1,External,agriculture
1,EVT_00010,<NA>,<NA>,2,External,healthcare
2,EVT_00012,1,3,2,Candidate,employment
3,EVT_00013,1,6,2,News,general
4,EVT_00014,1,7,2,News,agriculture


## 4. Check that the graph is local
An edge points from the voter who influences (`source_voter_id`) to the voter influenced (`target_voter_id`). The loader reports boundary edges it excluded; no excluded edge appears in the returned local graph.

In [4]:
local_ids = set(voters['voter_id'])
print('Every source is local:', edges['source_voter_id'].isin(local_ids).all())
print('Every target is local:', edges['target_voter_id'].isin(local_ids).all())
print('Excluded boundary edges:', constituency['diagnostics']['excluded_boundary_edge_count'])

Every source is local: True
Every target is local: True
Excluded boundary edges: 393


## What each returned item is for
- `voters`: agent attributes and starting opinions.
- `candidates`: candidate profiles and issue positions.
- `social_network`: directed, weighted influence ties within the constituency.
- `events`: local, regional, and state-wide event schedule.
- `diagnostics`: counts and geography, including excluded boundary edges.

These inputs are synthetic assumptions. The simulation code will decide how agents update opinions, respond to events, turn out, and vote.